# Chapter 7: Search In-Depth

## Topic 5: `classify_by_keyword_rules()` Reframed — Hand-Rolled BM25 Side by Side

---

### 1. Concept and Intuition

**What this topic actually is:**

- Not a new algorithm — a deliberate re-reading of code you already wrote in Chapter 1
- `classify_by_keyword_rules()` was built as a fast, zero-cost, zero-training rule engine for email classification
- Every topic in this chapter has referenced it as "the crudest possible form of sparse retrieval" — this topic makes that claim precise, term by term, rather than asserting it
- The goal: understand exactly which parts of BM25's mathematics your rule engine already approximates, which parts it skips entirely, and why skipping them was the right call for a binary classifier even though it would be the wrong call for ranked retrieval

**The exact function, unchanged from Chapter 1:**

```python
def classify_by_keyword_rules(subject: str, content: str) -> str:
    text = (subject + " " + content).lower()
    has_fd_term = any(kw in text for group in FD_KEYWORD_GROUPS for kw in group)
    has_negation = any(neg in text for neg in NEGATION_PHRASES)
    if has_fd_term and not has_negation:
        return "FD"
    elif has_negation and not has_fd_term:
        return "Non-FD"
    elif has_fd_term and has_negation:
        return "Multiple Category"
    else:
        return "ambiguous"
```

- `FD_KEYWORD_GROUPS`: loaded from `fd_keyword_groups.txt` — 4 groups, derived from real term-frequency counts against FD-labeled rows in `fd_dataset_messy.csv`
- `NEGATION_PHRASES`: loaded from `fd_negation_phrases.txt` — 16 veto terms (loan, emi, insurance, card, login, otp, kyc, branch, statement, etc.)
- Both files are loaded as single, unsplit Documents via `text_loader.py` — chunking a keyword list would tear related terms apart and actively hurt matching, the one deliberate exception to this project's usual chunking rules

**Why this comparison matters beyond curiosity:**

- Every Lead-level retrieval interview eventually asks "why not just do keyword matching?" — this topic gives you the precise, quantified answer instead of a vague "BM25 is better"
- Understanding exactly which BM25 components your rule engine is missing tells you exactly what breaks, and lets you predict failure modes before they happen in production rather than discovering them after

---

### 2. The Term-by-Term Mapping

**Component 1 — Term presence: rule engine has this, in a weaker form**

- Rule engine: `kw in text` — Python substring containment, case-folded via `.lower()`
- BM25: `TF(t,d) > 0` — also just checks whether the term appears, but then goes further
- Both start from the same binary fact: does this term appear anywhere in this text?
- The rule engine stops here. BM25 uses this as only the first input to a larger formula.

**Component 2 — Term frequency weighting: rule engine has none**

- Rule engine: `any(...)` short-circuits on the *first* match — whether "premature" appears once or eleven times in an email, the rule engine's `has_fd_term` flag is identically `True`
- BM25: `TF(t,d) × (k1+1) / (TF(t,d) + k1)` — frequency matters, with diminishing returns (Topic 1's saturation curve)
- Practical consequence: an email that mentions "loan" once as a passing aside ("I also have a loan with you") gets vetoed by `has_negation` exactly as hard as an email entirely about a loan EMI bounce — the rule engine cannot distinguish "mentioned once" from "this email is about it"

**Component 3 — Inverse document frequency (term rarity): rule engine has none**

- Rule engine: every keyword in `fd_keyword_groups.txt` is treated as equally diagnostic — "byaj" (appears rarely, highly discriminating) counts exactly the same as "interest rate" (a more generic phrase) for the purposes of `has_fd_term`
- BM25: `IDF(t) = log((N-df(t)+0.5)/(df(t)+0.5)+1)` — rare terms get more weight automatically, learned from corpus statistics, not manually curated
- Practical consequence: the rule engine's discriminating power for each keyword was hand-tuned by a human reading term-frequency counts once during development (as noted in the file header: "Derived from real term-frequency counts in fd_dataset_messy.csv"); BM25 recomputes this automatically every time the corpus changes

**Component 4 — Document length normalisation: rule engine has none, and arguably doesn't need it**

- Rule engine: operates on single emails averaging ~31 words — length variance within this corpus is small enough that BM25's `b` parameter would have limited effect anyway
- BM25: `(1 - b + b × |d|/avgdl)` — corrects for documents of very different lengths, which matters far more for the 17-page multi-length RAG knowledge base (FAQ entries vs. multi-step SOPs) than for short customer emails
- This is the one BM25 component that is genuinely *less relevant* to the rule engine's original use case — classification of short, similarly-sized emails — even though it matters greatly for retrieval over the knowledge base

**Component 5 — Negation handling: rule engine has this, BM25 has no equivalent**

- Rule engine: `has_negation` is a second, independent signal that can *override* or combine with the FD signal — this is genuine domain logic that plain BM25 does not model at all
- BM25 has no concept of "this term, if present, should suppress relevance" — a naive BM25 score for a query would simply add up term contributions; there is no subtraction
- This is worth stating explicitly in interviews: the rule engine is not simply "weaker BM25" — it also has structured domain logic (negation-based routing to three distinct output classes) that a raw BM25 score does not provide out of the box. Reproducing this in a retrieval-ranking framework would require a different mechanism (e.g. negative-weighted terms, or a separate veto-classifier stage).

---

### 3. Internal Working — Running Both Side by Side

**Setup: a shared test case**

- Email: `"I have applied for a personal loan against my FD, please process fast"`
- Rule engine walkthrough:
  1. `text = "i have applied for a personal loan against my fd, please process fast"`
  2. `has_fd_term`: `"fd"` matches → `True` (via `FD_ACCOUNT_REFERENCE` group, if "fd" alone is present, or matched loosely depending on group definitions)
  3. `has_negation`: `"loan"` matches → `True`
  4. Both flags true → returns `"Multiple Category"`
- BM25 walkthrough (over the same email treated as a mini corpus of concept-terms):
  1. Tokenize: `["i","have","applied","for","a","personal","loan","against","my","fd","please","process","fast"]`
  2. `TF("loan", d) = 1`, `TF("fd", d) = 1` — both appear once
  3. If "loan" is rarer in the corpus than "fd" (very likely, since "fd" appears in ~40% of all emails per the EDA), `IDF("loan") > IDF("fd")` — BM25 would weight the loan signal more heavily
  4. BM25 alone produces a *relevance score*, not a classification — it has no mechanism to say "Multiple Category"; that decision layer would need to be built on top of BM25 scores (e.g. thresholds, or two separate BM25 indexes — one for FD terms, one for negation terms — with scores compared)

**What this walkthrough demonstrates:**

- The rule engine's `has_fd_term and has_negation` branching *is* a form of decision logic BM25 does not provide natively
- If you wanted to rebuild `classify_by_keyword_rules()` on top of BM25, the natural design is: two separate BM25 indexes (one over FD-signal terms, one over negation terms), score the email against both, then apply the same three-way branching logic using score thresholds instead of binary flags — this would add frequency and rarity weighting to the existing decision structure without discarding the domain logic that makes the rule engine effective

---

### 4. How It Is Implemented in This Project

**The rule engine's role does not change:**

- Chapter 1's `classify_by_keyword_rules()` remains part of the classification cascade — a fast, free, zero-latency first-pass filter, entirely unaffected by anything in this chapter
- This chapter's BM25 work (Topic 1 onward) is for a *different* task: ranked retrieval of policy document chunks for the RAG pipeline, not email classification
- These are two separate systems using overlapping ideas — the comparison in this topic is educational, not a refactor recommendation

**Where a BM25-style upgrade of the rule engine would actually make sense:**

- If the rule engine's false-positive rate on ambiguous emails (the 145 Non-FD emails containing FD keywords, from the EDA) needed to be reduced without adding an LLM call, replacing the binary `has_fd_term` flag with a BM25 relevance score compared against a threshold is the natural evolution
- This would require: building a small BM25 index where "documents" are the four keyword groups (concatenated), scoring each incoming email against each group, and using score magnitude instead of binary presence — catching the "loan mentioned once in passing" vs. "loan is the entire point of this email" distinction that the current rule engine cannot

**Code: a BM25-scored variant of the rule engine, side by side with the original**

```python
# Original — Chapter 1, unchanged
def classify_by_keyword_rules(subject: str, content: str) -> str:
    text = (subject + " " + content).lower()
    has_fd_term = any(kw in text for group in FD_KEYWORD_GROUPS for kw in group)
    has_negation = any(neg in text for neg in NEGATION_PHRASES)
    if has_fd_term and not has_negation:
        return "FD"
    elif has_negation and not has_fd_term:
        return "Non-FD"
    elif has_fd_term and has_negation:
        return "Multiple Category"
    else:
        return "ambiguous"

# BM25-scored variant — same decision structure, frequency + rarity aware
from rank_bm25 import BM25Okapi

def classify_by_bm25_rules(subject: str, content: str,
                            fd_bm25: BM25Okapi, negation_bm25: BM25Okapi,
                            threshold: float = 0.5) -> str:
    text_tokens = (subject + " " + content).lower().split()
    fd_score = sum(fd_bm25.get_scores(text_tokens))
    negation_score = sum(negation_bm25.get_scores(text_tokens))
    has_fd_term = fd_score > threshold
    has_negation = negation_score > threshold
    if has_fd_term and not has_negation:
        return "FD"
    elif has_negation and not has_fd_term:
        return "Non-FD"
    elif has_fd_term and has_negation:
        return "Multiple Category"
    else:
        return "ambiguous"
```

- Same four-way branching structure preserved — the domain logic that makes the rule engine effective is not discarded
- `has_fd_term` and `has_negation` are now threshold-based on a continuous score instead of a binary substring match — this is the specific, minimal change that adds frequency and rarity awareness

---

### 5. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

**Why the rule engine's simplicity was the right initial choice:**

- Zero dependencies beyond string operations — no model to load, no index to build, no GPU, sub-millisecond per email
- Fully deterministic and auditable — for a regulated NBFC domain, being able to point at the exact keyword that triggered a classification is a real compliance advantage over a black-box score
- The 4 keyword groups were derived from actual term-frequency analysis on labeled data, not guessed — this is a legitimate, evidence-based approach for an initial baseline

**Where the rule engine's simplicity becomes a real production cost:**

- 145 of 748 Non-FD emails in the corpus contain FD-related keywords — every one of these is a `has_fd_term=True` case that depends entirely on `has_negation` correctly catching the true topic; if the true topic uses vocabulary not in the 16-term negation list, this becomes a misclassification with zero graceful degradation (a binary flag has no "partial confidence" to fall back on)
- Every new product, every new phrasing pattern customers adopt, requires manually editing `fd_keyword_groups.txt` or `fd_negation_phrases.txt` — this is a maintenance burden that grows over time and requires someone to notice the gap first (usually via a misclassification review)

**Monitoring the rule engine layer specifically:**

- Track the fraction of emails landing in `"ambiguous"` over time — this is the rule engine's honest "I don't know" signal, and a rising ambiguous rate indicates either vocabulary drift (new products, new phrasing) or genuinely increasing query complexity
- Track keyword hit frequency per group — a keyword group with near-zero hits across months of production traffic likely reflects either declining relevance of that group's phrasing or evidence the keyword list needs revision
- Cross-reference rule engine output against final (possibly GenAI-assisted) classification to measure the rule engine's own precision/recall in production, not just at the original dev-set evaluation time

**Cost and latency:**

- Effectively zero marginal cost per email — string containment checks over two small keyword lists, microseconds per email
- This is precisely why it sits as the first, cheapest layer in the classification cascade — every email the rule engine confidently classifies never needs to reach a more expensive layer (prompted LLM call, RAG retrieval, etc.)

**Security:**

- No PII exposure risk specific to this layer — it operates purely on the email text already available in the pipeline, no external calls, no data leaving the local process
- The keyword and negation term files themselves are not sensitive (domain vocabulary, not customer data) but should still be version-controlled and reviewed like any other production logic, since a bad edit (e.g. a keyword that's too broad) has direct classification-accuracy consequences

---

### 6. Design Decisions, Trade-offs, Real-Time Dilemmas

**Binary flags vs. continuous scores — the core trade-off this topic surfaces:**

- Binary flags (current rule engine): simple, fast, fully interpretable, but brittle at the margin — "loan mentioned once" and "loan is the whole email" are indistinguishable
- Continuous scores (BM25-based variant): captures frequency and rarity nuance, but requires choosing a threshold — and threshold selection reintroduces exactly the kind of tuning-without-labeled-data problem BM25's k1/b parameters have (Topic 1, Section on design decisions)
- For this project's current scale and requirements, the binary rule engine's simplicity is likely still the right choice — the real question is whether the 145-email overlap problem is costing enough in downstream misclassification to justify the added complexity of maintaining a BM25 index alongside the existing keyword-list workflow

**Should the negation mechanism be preserved if BM25 is adopted?**

- Yes, unconditionally — this is domain logic that plain BM25 relevance scoring does not replace, only complement
- Any upgrade path must keep the three-way (FD / Non-FD / Multiple Category) branching intact; the upgrade is in *how* `has_fd_term` and `has_negation` are computed, not in discarding the branching structure itself

**When does the added complexity of a BM25-scored rule engine variant actually pay off?**

- When the ambiguous-rate or misclassification-rate monitoring (Section 5) shows the binary flag's brittleness is the actual bottleneck — not before
- When there is a labeled evaluation set large enough to properly tune the threshold parameter, avoiding the trap of picking a threshold that looks good on 5 examples and fails in production
- Premature adoption of the BM25-scored variant, without evidence the binary version is failing, adds index-maintenance overhead for a benefit that may not materialise

---

### 7. Alternatives and When to Use Each

**Pure binary rule engine (current, Chapter 1):**
- Best for: initial baseline, zero infrastructure, full auditability, situations where keyword presence alone is genuinely sufficient signal
- Use when: false-positive/false-negative rate from the 145-email overlap problem is acceptable relative to downstream correction cost (e.g. a human review queue catches the rest)

**BM25-scored rule engine variant (this topic's proposal):**
- Best for: retaining the interpretable three-way branching logic while adding frequency/rarity awareness, without the cost of a full ML classifier
- Use when: monitoring shows binary-flag brittleness is a measurable, recurring source of misclassification, and a labeled set exists to tune the threshold properly

**Full ML classifier (MuRIL, Chapter 1's actual production baseline):**
- Best for: maximum accuracy (0.97 FD Recall, the number every other approach in this project is measured against), when training data and infrastructure for a fine-tuned model are available
- Use when: the accuracy gap between rule-based approaches and a trained classifier has real business cost, and the infrastructure to serve a fine-tuned model already exists

**Prompted LLM classification (Chapter 2):**
- Best for: handling genuinely novel phrasing, sarcasm, and edge cases neither keyword rules nor a fixed-vocabulary classifier can generalise to, without retraining
- Use when: the case has already failed or been flagged uncertain by the cheaper layers (rule engine, MuRIL) — this project's own cascade design treats GenAI as Layer 4, the expensive last resort for the hard ~1-3% slice, not the first-line classifier

---

### 8. Common Mistakes and Production Failures

- **Treating the rule engine and BM25 as competing systems rather than complementary tools for different tasks** — the rule engine classifies emails; BM25 (Topics 1-4) retrieves document chunks; conflating them leads to inappropriate "just replace it with BM25" recommendations that ignore the negation-branching logic BM25 does not provide
- **Editing `fd_keyword_groups.txt` or `fd_negation_phrases.txt` without re-validating against the labeled dev set** — a keyword addition that looks reasonable can silently shift the FD/Non-FD/Multiple-Category balance in ways only visible in a full confusion-matrix re-run, not by inspection
- **Assuming BM25 term weighting would trivially "fix" the 145-email overlap problem** — many of those 145 emails likely contain FD keywords for legitimate but non-FD reasons (e.g. mentioning an FD only as context for an unrelated complaint); frequency/rarity weighting helps but does not eliminate every case, since the underlying issue is topical ambiguity in the email itself, not solely keyword weighting
- **Picking a BM25 relevance threshold without a labeled evaluation set** — exactly the same trap as picking k1/b without evaluation data (Topic 1); a threshold tuned on a handful of manually inspected examples will not generalise reliably
- **Forgetting that "ambiguous" is a legitimate, useful output, not a bug to eliminate entirely** — a rising ambiguous rate is valuable signal (vocabulary drift, new product launches); routing every ambiguous case to a more expensive downstream layer is often the correct design, not a failure of the rule engine

---

### 9. Lead-Level Interview Questions

**Basic:**

**Q: In one sentence, what is `classify_by_keyword_rules()` missing that BM25 provides?**

A: It has no notion of term frequency (how many times a keyword appears) or inverse document frequency (how rare/diagnostic a keyword is across the corpus) — every keyword match is treated as an equally weighted binary flag, whereas BM25 weights matches by both frequency (with saturation) and corpus-wide rarity.

**Q: Does BM25 have a direct equivalent to the rule engine's negation-based veto mechanism?**

A: No. BM25 produces a single relevance score per document from positively-weighted term matches; it has no built-in concept of a term that should suppress or override the score. The rule engine's `has_negation` branching is genuine domain logic layered on top of keyword matching, and any BM25-based redesign would need to explicitly reconstruct this — for example with two separate BM25 indexes (FD-signal terms and negation terms) compared against each other, as shown in this topic's code.

**Intermediate:**

**Q: 145 of 748 Non-FD emails in the corpus contain FD-related keywords. Would swapping the rule engine's binary flags for BM25 relevance scores eliminate this problem?**

A: Partially, not entirely. BM25 would help in cases where the FD keyword appears only once, in passing, and the email is genuinely dominated by other terms — the low term frequency combined with the FD keyword's low corpus-wide rarity (since "FD" appears in roughly 40% of the corpus) would produce a modest BM25 score, correctly signalling low relevance. But cases where an email is topically ambiguous — genuinely discussing both an FD and an unrelated issue with roughly equal emphasis — are a content-level ambiguity that keyword weighting alone cannot resolve; that requires either better negation-term coverage or escalation to a downstream classifier that understands sentence-level context, which is exactly why this project's cascade design still exists.

**Q: Why does document length normalisation (BM25's b parameter) matter far less for the rule engine's original use case than for RAG retrieval over the knowledge base?**

A: The rule engine classifies individual customer emails, which the EDA shows average approximately 31 words with relatively low length variance across the corpus. BM25's b parameter primarily corrects for large differences in document length within a single corpus — it matters much more when comparing a 3-sentence FAQ answer against an 8-step SOP document (the RAG knowledge base's actual length variance) than when comparing one 25-word email against another 35-word email, where the effect is comparatively small.

**Advanced:**

**Q: Design a migration path from the current binary rule engine to a BM25-scored variant without breaking the existing three-class output contract the rest of the pipeline depends on.**

A: Preserve the exact function signature and the three-way branching structure (`FD` / `Non-FD` / `Multiple Category` / `ambiguous`) unchanged, since downstream code depends on this contract. Internally, replace the two binary flags with threshold comparisons against BM25 relevance scores computed from two separate indexes — one built from the FD keyword groups treated as a small corpus of "documents" (one per group, or one combined), one from the negation phrases similarly. Validate the new implementation against the exact same labeled dev set used in Chapter 2's evaluation notebook before deployment, comparing not just aggregate accuracy but the specific confusion-matrix cells (FD→Non-FD errors, Non-FD→FD errors) to confirm the change actually reduces the 145-email overlap problem rather than shifting errors elsewhere. Deploy behind a feature flag or shadow-mode comparison against the existing binary rule engine in production before fully cutting over, since a threshold that looks correct on the dev set can behave differently on the full, more varied production traffic distribution.

**Q: A colleague argues the rule engine should simply be deleted now that BM25 hybrid retrieval (Topic 4) exists in the pipeline. How do you respond?**

A: These solve different problems and operate at different pipeline stages. The rule engine classifies incoming emails into FD/Non-FD/Multiple Category as a cheap first-pass filter before any retrieval or generation happens at all — deleting it would mean every email, including the large fraction the rule engine currently classifies correctly and instantly, would need to reach a more expensive downstream stage (RAG retrieval plus LLM generation) just to get a label. Topic 4's hybrid BM25+dense retrieval operates on a completely different task: finding relevant policy document chunks for the RAG generation layer, and only runs for emails that have already been identified as needing a substantive, context-grounded answer. Removing the rule engine would increase latency and cost for every single email in production, for no accuracy benefit on the cases it already handles correctly — the two systems belong at different stages of the same cascade, not in competition.

**Scenario-based:**

**Q: Six months into production, the ambiguous-classification rate for `classify_by_keyword_rules()` has risen from 3% to 11% of daily volume. Walk through your diagnosis and response.**

A: First, pull a sample of the newly-ambiguous emails and read them directly — the most likely explanations are either a new product launch introducing vocabulary not present in `fd_keyword_groups.txt` or `fd_negation_phrases.txt` (analogous to the "Smart Saver FD" novel-product-name problem raised earlier in this chapter), or a genuine shift in how customers phrase requests (seasonal, campaign-driven, or influenced by a UI change directing certain phrasing). Second, quantify whether this rising ambiguous rate is actually harmful — if the downstream cascade correctly handles ambiguous cases via a more expensive but accurate layer (MuRIL or prompted GenAI), rising ambiguous rate may simply mean the correct cases are being routed to a slower path, which is a cost problem, not an accuracy problem; distinguish these before proposing a fix. Third, if a specific missing vocabulary pattern is identified, update the keyword or negation lists and re-validate against the full labeled dev set (not just the new examples) to confirm no regression on previously-correct cases, using the same confusion-matrix discipline established in Chapter 2's evaluation notebook. Fourth, if the drift is broad and not reducible to a small vocabulary patch, this is the trigger point discussed in the design-decisions section for considering the BM25-scored variant, since a shifting query distribution is exactly the scenario where frequency/rarity-aware scoring outperforms static binary flags.

---

### 10. Hidden Concepts and Prerequisites

**The rule engine is a specific instance of a "boolean retrieval model" — a named concept in classical IR:**

- Boolean retrieval (AND/OR/NOT over term presence) predates TF-IDF and BM25 historically, and the rule engine's `has_fd_term and not has_negation` logic is a textbook boolean retrieval expression
- Knowing this term lets you correctly place the rule engine in the historical IR lineage during an interview: boolean retrieval → TF-IDF → BM25 → learned sparse (SPLADE) → dense → hybrid, which is the exact arc this chapter has walked

**Precision-recall trade-offs are implicit in the current threshold-free design:**

- Because `has_fd_term` and `has_negation` are binary, the rule engine has no tunable precision-recall trade-off knob at all — it is locked to whatever balance the keyword lists happen to produce
- A BM25-scored or ML-based approach reintroduces this knob (via threshold or classifier confidence cutoff), which is a meaningful capability the current design deliberately trades away for simplicity and auditability

**The 4 keyword groups reflect a form of manual feature engineering:**

- Selecting and grouping `fd_keyword_groups.txt` terms from real term-frequency counts is, in essence, hand-crafted feature selection — the same underlying goal as what a trained model's embedding space or a BM25 IDF calculation achieves automatically
- Recognising this connects the rule engine conceptually to Chapter 0's classical ML foundations (feature engineering before deep learning) rather than treating it as an unrelated ad hoc script

---

### 11. Revision Summary

> `classify_by_keyword_rules()` is a boolean retrieval model: binary term presence combined with domain-specific negation logic, with no term-frequency weighting, no inverse-document-frequency rarity weighting, and no document-length normalisation. BM25 (Topic 1) supplies exactly the frequency and rarity weighting the rule engine lacks, but has no native equivalent to the rule engine's negation-based three-way branching — that domain logic must be explicitly reconstructed on top of BM25 scores (e.g. via two separate indexes and threshold comparisons) if a BM25-based upgrade is ever pursued. The rule engine and BM25-based retrieval (Topics 1-4) solve different problems at different pipeline stages — email classification vs. document chunk retrieval — and are not competing systems. The 145-email keyword-overlap problem in the corpus is a real, measured limitation of binary flags, but a BM25-scored variant should only be adopted after monitoring confirms this is an active production cost and a labeled evaluation set exists to tune thresholds properly, following the same evidence-based discipline used everywhere else in this project.

---